In [1]:
!pip install -q transformers accelerate bitsandbytes requests

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import os
import gc 
from google.colab import drive

In [3]:
# 1. Setup
drive.mount('/content/drive')
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DRIVE_ROOT = '/content/drive/MyDrive/Project'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
def run_rq3_context_inference(model_id="deepseek-ai/deepseek-coder-7b-instruct-v1.5"):
    # 1. Path Configuration
    # 'formal' exclusively for RQ3 to test if an Auditor is fooled by comments
    prompt_path = os.path.join(DRIVE_ROOT, 'prompts/formal_persona.txt') 
    input_path = os.path.join(DRIVE_ROOT, 'data/deceptive_experimental_set.csv')
    output_path = os.path.join(DRIVE_ROOT, 'results/rq3_deceptive_results.csv')

    # 2. Resume Logic (Robust Version)
    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        try:
            existing_df = pd.read_csv(output_path)
            # Ensure we only have unique indices
            existing_df = existing_df.drop_duplicates(subset=['index'])
            processed_indices = set(existing_df['index'].astype(int).tolist())
            results_list = existing_df.to_dict('records')
            print(f"--- Resuming RQ3: {len(processed_indices)} samples already found. ---")
        except pd.errors.EmptyDataError:
            print("--- Found empty results file. Starting from scratch. ---")
            processed_indices = set()
            results_list = []
    else:
        processed_indices = set()
        results_list = []
        # If the file exists but is empty, delete it to be safe
        if os.path.exists(output_path):
            os.remove(output_path)
        print(f"--- Starting RQ3 Deceptive Inference from scratch. ---")

        
    # 3. Data & Prompt Loading
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Missing deceptive dataset at: {input_path}")
        
    df = pd.read_csv(input_path)
    df['index'] = df['index'].astype(int)
    
    with open(prompt_path, 'r', encoding='utf-8') as f:
        system_instruction = f.read().strip()

    # 4. Model Loading (Only if work is remaining)
    if len(processed_indices) >= len(df):
        print("All samples already processed.")
        return

    print(f"--- Loading Model: {model_id} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # 5. Inference Loop
    print(f"--- Running Deceptive Context Inference ---")
    new_count = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        curr_idx = int(row['index'])
        if curr_idx in processed_indices:
            continue

        # In RQ3, the 'code' already contains the injected deceptive comment
        code_with_comment = row['code']
        prompt = f"{system_instruction}\n\nCODE:\n{code_with_comment}\n\nExplanation:"
        
        try:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2800).to("cuda")
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    tokenizer=tokenizer, 
                    max_new_tokens=300,         
                    temperature=0.1, 
                    do_sample=True,
                    repetition_penalty=1.1,    
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    stop_strings=["\n\n", "Note:"] 
                )
            
            explanation = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            
            results_list.append({
                'index': curr_idx,
                'cwe': row.get('cwe', 'Unknown'),
                'deception_type': row.get('deception_type', 'N/A'),
                'injected_comment': row.get('injected_comment', 'N/A'),
                'generated_explanation': explanation.strip()
            })
            
            new_count += 1
            
            # Frequent saves to prevent data loss in Colab
            if new_count % 2 == 0:
                pd.DataFrame(results_list).to_csv(output_path, index=False)
                torch.cuda.empty_cache()
                if new_count % 10 == 0:
                    print(f" [Heartbeat] {new_count} new samples processed.")

        except Exception as e:
            print(f"\nError at index {curr_idx}: {e}")
            continue

    # Final Save
    pd.DataFrame(results_list).to_csv(output_path, index=False)
    print(f"--- Success! RQ3 Results saved to: {output_path} ---")
    
    # Cleanup
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [5]:
if __name__ == "__main__":
    run_rq3_context_inference()

--- Found empty results file. Starting from scratch. ---
--- Loading Model: deepseek-ai/deepseek-coder-7b-instruct-v1.5 ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

--- Running Deceptive Context Inference ---


 16%|█▌        | 10/64 [04:49<27:42, 30.79s/it]

...

 [Heartbeat] 50 new samples processed.


 94%|█████████▍| 60/64 [27:19<01:47, 26.76s/it]

 [Heartbeat] 60 new samples processed.


100%|██████████| 64/64 [29:16<00:00, 27.45s/it]


--- Success! RQ3 Results saved to: /content/drive/MyDrive/Project/results/rq3_deceptive_results.csv ---
